# 06a · Build county factor table (public data, 87 MN counties)

**Goal:** one shared, documented table of **county characteristics** from public data, used by
- **Q1 extension**: which county characteristics explain the mismatch between Feeding America food-insecurity estimates and food shelf activity;
- **Q6**: need index (PCA / equal weights) and need-vs-demand divergence score.

This notebook **does not analyse** anything. It downloads (or reads cached copies), cleans, validates and saves.

**Outputs** (`Data/processed/`)
- `county_factors_2022_2024.csv`: 1 row = county × year (87 × 3 = 261). Factors that change every year.
- `county_factors_static.csv`: 1 row = county (87). Factors with a single vintage (SVI 2022, food access 2019 / 2025, County Health Rankings 2025).
- `county_factors_data_dictionary.csv`: every column, unit, source, vintage, grain.

**Not repeated here** (already in `q1_county_profiles.csv` / other processed files): ACS poverty (S1701), no-vehicle rate, RUCC, % rural, density, food-shelf sites per 10k, SNAP rate, distance to food shelf, commuting.

**Raw downloads** go to `Data/external/public_factors/raw/<source>/`. If a file is already there it is **not** downloaded again, so re-running gives the same result. Raw files are read with pandas only.

**August 2026 rule:** no source here has monthly data. BLS is requested for 2022–2024 only and only annual averages are kept; every table is checked for years ≥ 2026 before saving (none found).

**ACS columns (Census API) are added afterwards by `06b_acs_factors.ipynb`.** Re-running this notebook rewrites `county_factors_2022_2024.csv` without them, so re-run `06b` after it.

In [1]:
import io, json, re, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import requests

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "Data/external/public_factors/raw"
OUT = ROOT / "Data/processed"
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}
YEARS = [2022, 2023, 2024]

def fetch(url, path):
    # Download url to path unless a cached copy exists. Returns the path.
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(url, headers=UA, timeout=600)
    r.raise_for_status()
    path.write_bytes(r.content)
    return path

# Reference list of the 87 counties (fips as text) from the MMG file
mmg = pd.read_csv(OUT / "mmg_mn_county_2022_2024.csv", dtype={"fips": str})
counties = mmg[["fips", "county"]].drop_duplicates().sort_values("fips").reset_index(drop=True)
FIPS = set(counties.fips)
assert len(counties) == 87
print(len(counties), "counties, e.g.", counties.head(3).values.tolist())

87 counties, e.g. [['27001', 'Aitkin'], ['27003', 'Anoka'], ['27005', 'Becker']]


## 1 · Population (Census Population Estimates Program, Vintage 2024)

- **File:** `co-est2024-alldata.csv`, July 1 population estimates 2020–2024 for every county.
- **Why:** an annual population denominator for per-capita rates. ACS 5-year populations are 5-year averages; PEP is a July 1 point estimate for that year.
- Vintage 2024 revises earlier years, so 2022 and 2023 here may differ slightly from older PEP releases.

In [2]:
p = pd.read_csv(fetch("https://www2.census.gov/programs-surveys/popest/datasets/2020-2024/counties/totals/co-est2024-alldata.csv",
                      RAW / "popest/co-est2024-alldata.csv"), encoding="latin1", dtype={"STATE": str, "COUNTY": str})
p = p[(p.STATE == "27") & (p.COUNTY != "000")].copy()
p["fips"] = p.STATE + p.COUNTY
pop = p.melt(id_vars="fips", value_vars=[f"POPESTIMATE{y}" for y in YEARS], var_name="year", value_name="pop_pep")
pop["year"] = pop.year.str[-4:].astype(int)
print(pop.shape, pop.groupby("year").pop_pep.sum().to_dict())

(261, 3) {2022: 5721621, 2023: 5753048, 2024: 5793151}


## 2 · Unemployment rate (BLS Local Area Unemployment Statistics)

- **Series:** `LAUCN27xxx0000000003` = unemployment rate, county, not seasonally adjusted; we keep the **annual average** (period `M13`).
- The BLS flat files (`laucntyYY.xlsx`) return **403 (blocked)** to scripts, so we use the public **API v2 without a key** (max 25 series per request → 4 requests). Responses are cached as JSON.
- Requested years: 2022–2024 only. Monthly rows in the response (all 2022–2024) are dropped; only `M13` is kept.

In [3]:
codes = sorted(FIPS)
series = [f"LAUCN{f}0000000003" for f in codes]
rows = []
for i in range(0, len(series), 25):
    batch = series[i:i+25]
    cache = RAW / f"bls_laus/laus_unemp_rate_2022_2024_batch{i//25+1}.json"
    if not cache.exists():
        cache.parent.mkdir(parents=True, exist_ok=True)
        r = requests.post("https://api.bls.gov/publicAPI/v2/timeseries/data/",
                          json={"seriesid": batch, "startyear": "2022", "endyear": "2024", "annualaverage": True},
                          headers=UA, timeout=120)
        r.raise_for_status()
        js = r.json()
        assert js["status"] == "REQUEST_SUCCEEDED", js.get("message")
        cache.write_text(json.dumps(js))
    js = json.loads(cache.read_text())
    for s in js["Results"]["series"]:
        for d in s["data"]:
            rows.append({"fips": s["seriesID"][5:10], "year": int(d["year"]), "period": d["period"], "value": d["value"]})
laus = pd.DataFrame(rows)
print("rows in API response:", len(laus), "| max year:", laus.year.max())
laus = laus[(laus.period == "M13") & laus.year.isin(YEARS)]   # annual average only
laus = laus.assign(unemployment_rate=pd.to_numeric(laus.value, errors="coerce"))[["fips", "year", "unemployment_rate"]]
print(laus.shape, laus.groupby("year").unemployment_rate.describe()[["min", "50%", "max"]])

rows in API response: 3393 | max year: 2024
(261, 3)       min  50%  max
year               
2022  1.8  2.6  6.8
2023  2.1  3.0  6.7
2024  2.1  3.3  7.0


## 3 · Poverty and median household income (Census SAIPE)

- **Small Area Income and Poverty Estimates**: model-based **single-year** estimates (ACS + tax + SNAP records), one file per year.
- **Why SAIPE in addition to ACS S1701:** the ACS file we have is 5-year (2018–22 … 2020–24), which barely moves. SAIPE gives a separate estimate for each year and is what USDA/ED use for small areas. It is also the only keyless county median income series by year.
- Read from the fixed-width `.txt` version (the `.xls` needs the `xlrd` package, which isn't installed). Columns are whitespace separated; county rows have 21 numbers before the name.
- 90% confidence intervals are kept as a half-width (`_moe`).

In [4]:
def read_saipe(y):
    yy = str(y)[2:]
    f = fetch(f"https://www2.census.gov/programs-surveys/saipe/datasets/{y}/{y}-state-and-county/est{yy}all.txt",
              RAW / f"saipe/est{yy}all.txt")
    out = []
    for line in f.read_text(encoding="latin1").splitlines():
        t = line.split()
        if len(t) < 25 or t[0] != "27" or t[1] == "0":
            continue
        v = [float(x) for x in t[2:23]]
        out.append({"fips": "27" + t[1].zfill(3), "year": y,
                    "saipe_poverty_rate": v[3], "saipe_poverty_rate_moe": (v[5] - v[4]) / 2,
                    "saipe_child_poverty_rate": v[9], "saipe_child_poverty_rate_moe": (v[11] - v[10]) / 2,
                    "median_hh_income": v[18], "median_hh_income_moe": (v[20] - v[19]) / 2,
                    "_file": t[-2]})
    d = pd.DataFrame(out)
    assert (d._file == f"est{yy}all.txt").all()   # the right year's file
    return d.drop(columns="_file")

saipe = pd.concat([read_saipe(y) for y in YEARS], ignore_index=True)
print(saipe.shape)
saipe.groupby("year")[["saipe_poverty_rate", "median_hh_income"]].median()

(261, 8)


,saipe_poverty_rate,median_hh_income
year,,
2022,9.9,68888.0
2023,10.3,71573.0
2024,10.3,74003.0


## 4 · Grocery / food stores per 10k residents (Census County Business Patterns)

- **CBP county file**, establishment counts by NAICS code (mid-March of each year). 2022 and 2023 are published; **2024 is not yet** (404), so 2024 is blank, not copied.
- CBP 2022–2023 still use **NAICS 2017** codes. `445110` = supermarkets & other grocery stores (except convenience). `445` = all food & beverage stores (grocery, convenience, specialty food, liquor).
- ⚠️ **Since 2017 CBP does not publish a row with fewer than 3 establishments** (the smallest published `est` is 3, checked below). So a missing row means **0, 1 or 2 stores**, *not* zero. We leave those counts **blank** and set `grocery_count_lt3 = True`. If a Q6 index needs a number, impute 1 (midpoint) and say so.
- Supercenters (`452311`) were dropped: published for only 16 of 87 counties.
- Per 10k uses PEP July 1 population of the same year.

In [5]:
cbp_rows = []
for y in [2022, 2023]:
    yy = str(y)[2:]
    z = zipfile.ZipFile(fetch(f"https://www2.census.gov/programs-surveys/cbp/datasets/{y}/cbp{yy}co.zip", RAW / f"cbp/cbp{yy}co.zip"))
    c = pd.read_csv(z.open(z.namelist()[0]), dtype=str, encoding="latin1")
    c.columns = c.columns.str.lower()
    c = c[c.fipstate == "27"]
    c["fips"] = "27" + c.fipscty.str.zfill(3)
    for code_, name in [("445110", "n_grocery_stores"), ("445///", "n_food_bev_stores")]:
        s = c[c.naics == code_].set_index("fips").est.astype(int)
        assert s.min() >= 3                                    # confirms the <3 rule
        for f in sorted(FIPS):
            cbp_rows.append({"fips": f, "year": y, "var": name, "n": s.get(f, np.nan)})
cbp = pd.DataFrame(cbp_rows).pivot_table(index=["fips", "year"], columns="var", values="n", dropna=False).reset_index()
cbp.columns.name = None
cbp["grocery_count_lt3"] = cbp.n_grocery_stores.isna()
cbp = cbp.merge(pop, on=["fips", "year"])
cbp["grocery_stores_per_10k"] = cbp.n_grocery_stores / cbp.pop_pep * 1e4
cbp["food_bev_stores_per_10k"] = cbp.n_food_bev_stores / cbp.pop_pep * 1e4
cbp = cbp.drop(columns="pop_pep")
print(cbp.groupby("year")[["n_grocery_stores", "n_food_bev_stores"]].sum())
print("counties with < 3 grocery stores (not published):", cbp.groupby("year").grocery_count_lt3.sum().to_dict())
print("counties with < 3 food & beverage stores:", cbp[cbp.n_food_bev_stores.isna()][["fips", "year"]].values.tolist())

      n_grocery_stores  n_food_bev_stores
year                                     
2022             973.0             2465.0
2023             966.0             2444.0
counties with < 3 grocery stores (not published): {2022: 17, 2023: 18}
counties with < 3 food & beverage stores: [['27125', 2023], ['27167', 2022], ['27167', 2023]]


## 5 · Merge the year-varying table and validate

In [6]:
grid = pd.MultiIndex.from_product([sorted(FIPS), YEARS], names=["fips", "year"]).to_frame(index=False)
yearly = (grid.merge(counties, on="fips")
              .merge(pop, on=["fips", "year"], how="left")
              .merge(laus, on=["fips", "year"], how="left")
              .merge(saipe, on=["fips", "year"], how="left")
              .merge(cbp, on=["fips", "year"], how="left"))
for c in ["n_grocery_stores", "n_food_bev_stores"]:
    yearly[c] = yearly[c].astype("Int64")
yearly["grocery_count_lt3"] = yearly.grocery_count_lt3.astype("boolean")   # blank in 2024 (no CBP yet)

assert len(yearly) == 261 and not yearly.duplicated(["fips", "year"]).any()
assert yearly.fips.str.fullmatch(r"27\d{3}").all() and set(yearly.fips) == FIPS
assert yearly.year.max() < 2026                                   # August-2026 rule (no 2026 at all)
assert yearly.unemployment_rate.between(0, 25).all()
assert yearly.saipe_poverty_rate.between(0, 60).all()
assert yearly.median_hh_income.between(20000, 200000).all()
print("missing values per column:\n", yearly.isna().sum()[yearly.isna().sum() > 0])
yearly.describe().T[["count", "min", "50%", "max"]].round(2)

missing values per column:
 n_food_bev_stores           90
n_grocery_stores           122
grocery_count_lt3           87
grocery_stores_per_10k     122
food_bev_stores_per_10k     90
dtype: int64


,count,min,50%,max
year,261.0,2022.0,2023.0,2024.0
pop_pep,261.0,3134.0,22045.0,1273334.0
unemployment_rate,261.0,1.8,3.0,7.0
saipe_poverty_rate,261.0,4.5,10.2,19.3
saipe_poverty_rate_moe,261.0,0.7,1.9,3.6
saipe_child_poverty_rate,261.0,3.5,11.3,31.1
saipe_child_poverty_rate_moe,261.0,1.1,3.4,7.1
median_hh_income,261.0,52926.0,71892.0,129437.0
median_hh_income_moe,261.0,1696.0,5696.0,9996.0
n_food_bev_stores,171.0,3.0,13.0,548.0


## 6 · CDC/ATSDR Social Vulnerability Index 2022 (static)

- Minnesota county file. Built from **ACS 2018–2022 5-year** estimates. Percentile ranks (`RPL_*`) are ranked **within Minnesota** (state file), 0–1, higher = more vulnerable.
- We keep the 4 theme percentiles + overall, and the raw **percentages** (`EP_*`) that the team wants and don't already have (uninsured, disability, limited English, single parent, minority, mobile homes, crowding, age 65+, age 17−, housing cost burden, no internet, race shares incl. AIAN). `-999` = missing → blank.
- SVI's no-vehicle and poverty columns are skipped (already in the ACS files).

In [7]:
svi = pd.read_csv(fetch("https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/Minnesota_COUNTY.csv",
                        RAW / "svi/Minnesota_COUNTY_2022.csv"), dtype={"FIPS": str})
svi = svi.replace(-999, np.nan)
svi_map = {"RPL_THEMES": "svi_overall_pctile", "RPL_THEME1": "svi_socioeconomic_pctile", "RPL_THEME2": "svi_household_pctile",
           "RPL_THEME3": "svi_minority_language_pctile", "RPL_THEME4": "svi_housing_transport_pctile",
           "EP_POV150": "pct_below_150_poverty", "EP_UNEMP": "pct_unemployed_acs", "EP_HBURD": "pct_housing_cost_burden_lowinc",
           "EP_NOHSDP": "pct_no_hs_diploma", "EP_UNINSUR": "pct_uninsured", "EP_AGE65": "pct_age_65plus",
           "EP_AGE17": "pct_age_17under", "EP_DISABL": "pct_disability", "EP_SNGPNT": "pct_single_parent_hh",
           "EP_LIMENG": "pct_limited_english", "EP_MINRTY": "pct_minority", "EP_MUNIT": "pct_multiunit_housing",
           "EP_MOBILE": "pct_mobile_homes", "EP_CROWD": "pct_crowded_housing", "EP_GROUPQ": "pct_group_quarters",
           "EP_NOINT": "pct_no_internet", "EP_AIAN": "pct_aian", "EP_HISP": "pct_hispanic", "EP_AFAM": "pct_black",
           "EP_ASIAN": "pct_asian"}
svi = svi.rename(columns={"FIPS": "fips"})[["fips"] + list(svi_map)].rename(columns=svi_map)
assert set(svi.fips) == FIPS and len(svi) == 87
svi.describe().T[["count", "min", "50%", "max"]].round(2)

,count,min,50%,max
svi_overall_pctile,87.0,0.0,0.5,1.0
svi_socioeconomic_pctile,87.0,0.0,0.5,1.0
svi_household_pctile,87.0,0.0,0.5,1.0
svi_minority_language_pctile,87.0,0.0,0.5,1.0
svi_housing_transport_pctile,87.0,0.0,0.5,1.0
pct_below_150_poverty,87.0,6.8,18.2,36.0
pct_unemployed_acs,87.0,1.0,3.7,10.3
pct_housing_cost_burden_lowinc,87.0,13.1,20.2,27.9
pct_no_hs_diploma,87.0,2.8,6.7,20.6
pct_uninsured,87.0,2.4,4.9,13.7


## 7 · Food access: USDA ERS Food Access Research Atlas (2019) and SNAP-retailer access map (2025)

Both are **census-tract** files; we add tracts up to the county.

**(a) Food Access Research Atlas, 2019 vintage** (now called the *Large Retailer Access Map*; 2010 tracts, 2010 Census population, store list from **2019**; released April 2021). "Store" = supermarket, supercenter or large grocery.
- `fara_pct_low_access_1_10`: % of population more than **1 mile (urban) / 10 miles (rural)** from a store = Σ`LAPOP1_10` / Σ`Pop2010`.
- `fara_pct_lowinc_low_access_1_10`: % of population that is **low-income and** low access = Σ`LALOWI1_10` / Σ`Pop2010`.
- `fara_pct_pop_in_lila_tracts`: % of population living in a tract flagged low-income & low-access (1/10 mi) (`LILATracts_1And10`).
- `fara_pct_hh_novehicle_low_access`: % of housing units with no vehicle **and** beyond 1 mi (urban tracts) / 10 mi (rural tracts) = Σ(`lahunv1` or `lahunv10`) / Σ`OHU2010`.

**(b) SNAP-authorized Retailer Access Map, 2025** (2020 tracts and population; distance to **any SNAP-authorized retailer**, incl. small stores; straight-line; released **July 2026**, before the cutoff and containing no 2026-08+ data). Same four measures with `sram_` prefix, plus `pct_pop_tribal_area_approx`: ERS gives the **share of each tract's area** inside a tribal subdivision (reservation / trust land); we weight it by tract population (assumes people are spread evenly within a tract, so it's approximate).

Both use Σ counts / Σ population (population-weighted), not an average of tract percentages. Blank tract values = 0 people affected (ERS leaves cells blank when not applicable), checked below.

In [8]:
z = zipfile.ZipFile(fetch("https://www.ers.usda.gov/media/5627/2019-large-retailer-access-map-lram-formerly-known-as-the-food-access-research-atlas-fara-data.zip?v=75515",
                          RAW / "ers_fara/fara2019.zip"))
fa = pd.read_csv(z.open("Food Access Research Atlas.csv"), dtype={"CensusTract": str}, low_memory=False)
fa["fips"] = fa.CensusTract.str.zfill(11).str[:5]
fa = fa[fa.fips.str.startswith("27")].copy()
print("MN tracts (2010):", len(fa), "| tracts with blank LAPOP1_10:", fa.LAPOP1_10.isna().sum())
num = ["Pop2010", "OHU2010", "LAPOP1_10", "LALOWI1_10", "lahunv1", "lahunv10"]
fa[num] = fa[num].apply(pd.to_numeric, errors="coerce").fillna(0)
fa["hunv_la"] = np.where(fa.Urban == 1, fa.lahunv1, fa.lahunv10)
fa["pop_lila"] = fa.Pop2010 * (fa.LILATracts_1And10 == 1)
g = fa.groupby("fips")[["Pop2010", "OHU2010", "LAPOP1_10", "LALOWI1_10", "hunv_la", "pop_lila"]].sum()
fara = pd.DataFrame({
    "fara_pct_low_access_1_10": 100 * g.LAPOP1_10 / g.Pop2010,
    "fara_pct_lowinc_low_access_1_10": 100 * g.LALOWI1_10 / g.Pop2010,
    "fara_pct_pop_in_lila_tracts": 100 * g.pop_lila / g.Pop2010,
    "fara_pct_hh_novehicle_low_access": 100 * g.hunv_la / g.OHU2010,
}).reset_index()
assert set(fara.fips) == FIPS
fara.describe().T[["min", "50%", "max"]].round(1)

MN tracts (2010): 1334 | tracts with blank LAPOP1_10: 410


,min,50%,max
fara_pct_low_access_1_10,2.7,25.8,99.2
fara_pct_lowinc_low_access_1_10,0.7,6.0,26.2
fara_pct_pop_in_lila_tracts,0.0,19.4,100.0
fara_pct_hh_novehicle_low_access,0.0,1.0,4.8


In [9]:
z = zipfile.ZipFile(fetch("https://www.ers.usda.gov/media/29395/2025-snap-authorized-retailer-access-map-sram-data.zip?v=56500",
                          RAW / "ers_sram/sram2025.zip"))
gen = pd.read_csv(z.open("SRAM General Tract Characteristics Data.csv"), dtype={"CensusTract20": str}, encoding="latin1", low_memory=False)
sd = pd.read_csv(z.open("SRAM Straight Line Distance Data.csv"), dtype={"CensusTract20": str}, encoding="latin1", low_memory=False)
gen["tract"] = gen.CensusTract20.str.zfill(11); sd["tract"] = sd.CensusTract20.str.zfill(11)
sr = gen[gen.tract.str.startswith("27")][["tract", "Urban", "POP2020", "OHU2020", "TractTribalArea"]].merge(
     sd[["tract", "SD_SRAM_LAPOP1_10", "SD_SRAM_LALOWI1_10", "SD_SRAM_LILATracts_1And10", "SD_SRAM_lahunv1", "SD_SRAM_lahunv10"]],
     on="tract", how="left", validate="1:1")
sr["fips"] = sr.tract.str[:5]
print("MN tracts (2020):", len(sr), "| tracts partly in a tribal area:", (pd.to_numeric(sr.TractTribalArea, errors="coerce") > 0).sum())
num = ["POP2020", "OHU2020", "SD_SRAM_LAPOP1_10", "SD_SRAM_LALOWI1_10", "SD_SRAM_lahunv1", "SD_SRAM_lahunv10"]
sr[num] = sr[num].apply(pd.to_numeric, errors="coerce").fillna(0)
sr["hunv_la"] = np.where(sr.Urban == 1, sr.SD_SRAM_lahunv1, sr.SD_SRAM_lahunv10)
sr["pop_lila"] = sr.POP2020 * (sr.SD_SRAM_LILATracts_1And10 == 1)
# TractTribalArea = % of the tract's AREA inside a tribal subdivision; population share assumes even density within the tract
sr["pop_tribal"] = sr.POP2020 * pd.to_numeric(sr.TractTribalArea, errors="coerce").fillna(0) / 100
g = sr.groupby("fips")[["POP2020", "OHU2020", "SD_SRAM_LAPOP1_10", "SD_SRAM_LALOWI1_10", "hunv_la", "pop_lila", "pop_tribal"]].sum()
sram = pd.DataFrame({
    "sram_pct_low_access_1_10": 100 * g.SD_SRAM_LAPOP1_10 / g.POP2020,
    "sram_pct_lowinc_low_access_1_10": 100 * g.SD_SRAM_LALOWI1_10 / g.POP2020,
    "sram_pct_pop_in_lila_tracts": 100 * g.pop_lila / g.POP2020,
    "sram_pct_hh_novehicle_low_access": 100 * g.hunv_la / g.OHU2020,
    "pct_pop_tribal_area_approx": 100 * g.pop_tribal / g.POP2020,
}).reset_index()
assert set(sram.fips) == FIPS
sram.describe().T[["min", "50%", "max"]].round(1)

MN tracts (2020): 1502 | tracts partly in a tribal area: 60


,min,50%,max
sram_pct_low_access_1_10,0.0,6.3,27.0
sram_pct_lowinc_low_access_1_10,0.0,4.7,24.1
sram_pct_pop_in_lila_tracts,0.0,0.0,41.5
sram_pct_hh_novehicle_low_access,0.0,0.3,2.0
pct_pop_tribal_area_approx,0.0,0.0,100.0


## 8 · County Health Rankings & Roadmaps, 2025 release (static)

- One analytic CSV per release; row 0 = names, row 1 = variable codes. We pick measures by **name**.
- **Each measure has its own underlying data years** (listed in the data dictionary), mostly ACS 2019–2023 for the 2025 release. That's why it's in the static file and not lined up by year.
- `Food Insecurity` is skipped on purpose: it **is** Map the Meal Gap (circular for Q1/Q6).
- Proportions are converted to % (×100) so units match the other files.
- `Living Wage` was dropped: blank for all MN counties in the 2025 file.

In [10]:
chr_ = pd.read_csv(fetch("https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025.csv",
                         RAW / "chr/analytic_data2025.csv"), header=0, skiprows=[1], dtype={"5-digit FIPS Code": str}, low_memory=False)
chr_["fips"] = chr_["5-digit FIPS Code"].str.zfill(5)
chr_ = chr_[chr_.fips.str.startswith("27") & ~chr_.fips.str.endswith("000")].copy()
# (CHR name, our column, multiply by)
chr_map = [
    ("Food Environment Index raw value", "chr_food_environment_index", 1),
    ("Limited Access to Healthy Foods raw value", "chr_pct_limited_access_healthy_food", 100),
    ("Severe Housing Cost Burden raw value", "chr_pct_severe_housing_cost_burden", 100),
    ("Children in Single-Parent Households raw value", "chr_pct_children_single_parent", 100),
    ("Broadband Access raw value", "chr_pct_broadband", 100),
    ("Uninsured raw value", "chr_pct_uninsured_under65", 100),
    ("Homeownership raw value", "chr_pct_homeownership", 100),
    ("Income Inequality raw value", "chr_income_inequality_ratio", 1),
    ("Children Eligible for Free or Reduced Price Lunch raw value", "chr_pct_children_frl_eligible", 100),
    ("Child Care Cost Burden raw value", "chr_pct_child_care_cost_burden", 100),
    ("Social Associations raw value", "chr_social_associations_per_10k", 1),
    ("Census Participation raw value", "chr_pct_census_participation", 100),
    ("Long Commute - Driving Alone raw value", "chr_pct_long_commute_alone", 100),
    ("% Disability: Functional Limitations raw value", "chr_pct_disability_functional", 100),
    ("% Not Proficient in English raw value", "chr_pct_not_proficient_english", 100),
    ("Feelings of Loneliness raw value", "chr_pct_loneliness", 100),
]
chr_out = chr_[["fips"]].copy()
for src, dst, k in chr_map:
    chr_out[dst] = pd.to_numeric(chr_[src], errors="coerce") * k
assert set(chr_out.fips) == FIPS and len(chr_out) == 87
chr_out.describe().T[["count", "min", "50%", "max"]].round(2)

,count,min,50%,max
chr_food_environment_index,87.0,6.30,8.60,9.80
chr_pct_limited_access_healthy_food,87.0,0.74,5.97,26.26
chr_pct_severe_housing_cost_burden,87.0,5.57,9.94,14.26
chr_pct_children_single_parent,87.0,7.84,16.98,39.72
chr_pct_broadband,87.0,79.08,86.94,94.44
chr_pct_uninsured_under65,87.0,3.26,5.80,11.01
chr_pct_homeownership,87.0,61.03,79.63,87.67
chr_income_inequality_ratio,87.0,3.19,4.08,5.10
chr_pct_children_frl_eligible,87.0,19.95,45.11,77.76
chr_pct_child_care_cost_burden,87.0,22.16,29.92,54.53


## 9 · Merge the static table and validate

In [11]:
static = (counties.merge(svi, on="fips", how="left").merge(fara, on="fips", how="left")
                  .merge(sram, on="fips", how="left").merge(chr_out, on="fips", how="left"))
assert len(static) == 87 and static.fips.is_unique and static.fips.str.fullmatch(r"27\d{3}").all()
pct_cols = [c for c in static.columns if c.startswith(("pct_", "fara_pct", "sram_pct", "chr_pct", "svi_"))]
lim = {c: (0, 1) if c.startswith("svi_") else (0, 100) for c in pct_cols}
for c, (lo, hi) in lim.items():
    assert static[c].dropna().between(lo, hi).all(), c
print("missing values per column:\n", static.isna().sum()[static.isna().sum() > 0])

missing values per column:
 Series([], dtype: int64)


## 10 · Cross-checks against files we already have

If our new numbers disagree wildly with existing ones, something is wrong with the parsing.

In [12]:
pov = pd.read_csv(OUT / "poverty_2022_2024_clean.csv", dtype={"county_geoid": str}).rename(columns={"county_geoid": "fips"})
veh = pd.read_csv(OUT / "vehicle_access_2022_2024_clean.csv", dtype={"county_geoid": str}).rename(columns={"county_geoid": "fips"})
x = yearly.merge(pov[["fips", "year", "poverty_rate"]], on=["fips", "year"])
print("SAIPE vs ACS 5-yr poverty, Pearson r by year:", {y: round(d.saipe_poverty_rate.corr(d.poverty_rate), 3) for y, d in x.groupby("year")})
print("SAIPE 1-yr vs PEP: population totals (2024):", int(yearly.query("year==2024").pop_pep.sum()))
x = static.merge(veh.query("year==2022")[["fips", "no_vehicle_rate"]], on="fips")
print("SVI unemployment (ACS 18-22) vs BLS 2022 r:", round(static.pct_unemployed_acs.corr(yearly.query('year==2022').set_index('fips').unemployment_rate.reindex(static.fips).reset_index(drop=True)), 3))
print("FARA vs SRAM low-access share r:", round(static.fara_pct_low_access_1_10.corr(static.sram_pct_low_access_1_10), 3))
print("SVI vs CHR uninsured r:", round(static.pct_uninsured.corr(static.chr_pct_uninsured_under65), 3))

SAIPE vs ACS 5-yr poverty, Pearson r by year: {2022: 0.881, 2023: 0.866, 2024: 0.854}
SAIPE 1-yr vs PEP: population totals (2024): 5793151
SVI unemployment (ACS 18-22) vs BLS 2022 r: 0.52
FARA vs SRAM low-access share r: 0.264
SVI vs CHR uninsured r: 0.768


## 11 · Save

In [13]:
# tidy: id columns first, round derived values (no information lost at these precisions)
yearly = yearly[["fips", "county", "year"] + [c for c in yearly.columns if c not in ("fips", "county", "year")]]
num_y = yearly.select_dtypes("float").columns
yearly[num_y] = yearly[num_y].round(3)
num_s = static.select_dtypes("float").columns
static[num_s] = static[num_s].round(4)
yearly.to_csv(OUT / "county_factors_2022_2024.csv", index=False)
static.to_csv(OUT / "county_factors_static.csv", index=False)
print(yearly.shape, static.shape)

(261, 16) (87, 52)


## 12 · Data dictionary

In [14]:
SRC = {
 "pep": ("US Census Bureau, Population Estimates Program, Vintage 2024 county totals (co-est2024-alldata)", "July 1 of year (V2024)"),
 "laus": ("US BLS, Local Area Unemployment Statistics, series LAUCN27xxx0000000003 (annual avg, not seas. adj.)", "annual average of year"),
 "saipe": ("US Census Bureau, Small Area Income and Poverty Estimates (estYYall)", "calendar year (single-year model estimate)"),
 "cbp": ("US Census Bureau, County Business Patterns (cbpYYco)", "mid-March of year; 2024 not yet released"),
 "svi": ("CDC/ATSDR Social Vulnerability Index 2022, Minnesota county file", "ACS 2018-2022 5-year"),
 "fara": ("USDA ERS Food Access Research Atlas / Large Retailer Access Map, 2019 vintage", "stores 2019; Census 2010 population & tracts; released Apr 2021"),
 "sram": ("USDA ERS Food Access Research Atlas, SNAP-authorized Retailer Access Map 2025, straight-line distance", "SNAP retailers 2025; Census 2020 population & tracts; released Jul 2026"),
 "chr": ("County Health Rankings & Roadmaps (UW Population Health Institute), 2025 analytic data", "2025 release; underlying years noted per row"),
 "id": ("Map the Meal Gap file (ids only)", "-"),
}
def row(col, defn, unit, src, file, note=""):
    s, v = SRC[src]
    return {"file": file, "column": col, "definition": defn + (f" [{note}]" if note else ""), "unit": unit,
            "source": s, "vintage": v, "geo_grain": "county" if src not in ("fara", "sram") else "census tract summed to county",
            "time_grain": "annual (2022-2024)" if file.endswith("2022_2024.csv") else "single vintage (static)"}
Y, S = "county_factors_2022_2024.csv", "county_factors_static.csv"
d = [row("fips", "5-digit county FIPS code, read as TEXT; join key", "text", "id", Y),
     row("county", "County name", "text", "id", Y),
     row("year", "Calendar year", "year", "id", Y),
     row("pop_pep", "Resident population, July 1", "persons", "pep", Y),
     row("unemployment_rate", "Unemployment rate, annual average", "% of labor force", "laus", Y),
     row("saipe_poverty_rate", "People in poverty, all ages", "% of people", "saipe", Y),
     row("saipe_poverty_rate_moe", "Half-width of 90% CI for saipe_poverty_rate", "percentage points", "saipe", Y),
     row("saipe_child_poverty_rate", "Children 0-17 in poverty", "% of children", "saipe", Y),
     row("saipe_child_poverty_rate_moe", "Half-width of 90% CI for saipe_child_poverty_rate", "percentage points", "saipe", Y),
     row("median_hh_income", "Median household income (nominal dollars of that year)", "USD", "saipe", Y),
     row("median_hh_income_moe", "Half-width of 90% CI for median_hh_income", "USD", "saipe", Y),
     row("n_food_bev_stores", "Establishments NAICS 445 food & beverage stores (grocery, convenience, specialty, liquor); blank = fewer than 3 (not published) or 2024", "count", "cbp", Y),
     row("n_grocery_stores", "Establishments NAICS 445110 supermarkets & other grocery stores (exc. convenience); blank = fewer than 3 (not published) or 2024 (not released)", "count", "cbp", Y),
     row("grocery_count_lt3", "TRUE = county has 0-2 grocery stores (CBP does not publish <3); blank in 2024", "TRUE/FALSE", "cbp", Y),
     row("food_bev_stores_per_10k", "n_food_bev_stores / pop_pep x 10,000", "stores per 10k residents", "cbp", Y),
     row("grocery_stores_per_10k", "n_grocery_stores / pop_pep x 10,000; blank where count <3 or 2024", "stores per 10k residents", "cbp", Y),
     row("fips", "5-digit county FIPS code, read as TEXT; join key", "text", "id", S),
     row("county", "County name", "text", "id", S)]
svi_def = {"svi_overall_pctile": "SVI overall percentile rank within MN (RPL_THEMES); higher = more vulnerable",
           "svi_socioeconomic_pctile": "SVI theme 1 socioeconomic status percentile within MN (RPL_THEME1)",
           "svi_household_pctile": "SVI theme 2 household characteristics percentile within MN (RPL_THEME2)",
           "svi_minority_language_pctile": "SVI theme 3 racial & ethnic minority status percentile within MN (RPL_THEME3)",
           "svi_housing_transport_pctile": "SVI theme 4 housing type & transportation percentile within MN (RPL_THEME4)",
           "pct_below_150_poverty": "Persons below 150% of poverty (EP_POV150)", "pct_unemployed_acs": "Civilian 16+ unemployed (EP_UNEMP), ACS not BLS",
           "pct_housing_cost_burden_lowinc": "Housing cost-burdened occupied units with annual income < $75,000 (30%+ of income on housing) (EP_HBURD)",
           "pct_no_hs_diploma": "Persons 25+ with no high school diploma (EP_NOHSDP)", "pct_uninsured": "Civilian noninstitutionalized persons with no health insurance (EP_UNINSUR)",
           "pct_age_65plus": "Persons aged 65 and older (EP_AGE65)", "pct_age_17under": "Persons aged 17 and younger (EP_AGE17)",
           "pct_disability": "Civilian noninstitutionalized persons with a disability (EP_DISABL)",
           "pct_single_parent_hh": "Single-parent households with children under 18 (EP_SNGPNT)",
           "pct_limited_english": "Persons 5+ who speak English 'less than well' (EP_LIMENG)",
           "pct_minority": "Minority (all except non-Hispanic White) (EP_MINRTY)", "pct_multiunit_housing": "Housing in structures with 10+ units (EP_MUNIT)",
           "pct_mobile_homes": "Mobile homes (EP_MOBILE)", "pct_crowded_housing": "Occupied units with more people than rooms (EP_CROWD)",
           "pct_group_quarters": "Persons in group quarters (EP_GROUPQ)", "pct_no_internet": "Households without an internet subscription (EP_NOINT)",
           "pct_aian": "American Indian & Alaska Native, non-Hispanic (EP_AIAN)", "pct_hispanic": "Hispanic or Latino (EP_HISP)",
           "pct_black": "Black/African American, non-Hispanic (EP_AFAM)", "pct_asian": "Asian, non-Hispanic (EP_ASIAN)"}
for c, t in svi_def.items():
    d.append(row(c, t, "percentile 0-1" if c.startswith("svi_") else "%", "svi", S))
fara_def = {"pct_low_access_1_10": "Population beyond 1 mi (urban tract) / 10 mi (rural tract) from nearest {store}",
            "pct_lowinc_low_access_1_10": "Population that is low-income AND beyond 1/10 mi from nearest {store}",
            "pct_pop_in_lila_tracts": "Population living in low-income & low-access (1/10 mi) tracts",
            "pct_hh_novehicle_low_access": "Housing units without a vehicle AND beyond 1 mi (urban) / 10 mi (rural) from nearest {store}"}
for c, t in fara_def.items():
    d.append(row("fara_" + c, t.format(store="supermarket/supercenter/large grocery"), "% of county population (housing units for hh measure)", "fara", S))
for c, t in fara_def.items():
    d.append(row("sram_" + c, t.format(store="SNAP-authorized retailer (any size), straight-line"), "% of county population (housing units for hh measure)", "sram", S))
d.append(row("pct_pop_tribal_area_approx", "Approx. population living inside an American Indian tribal subdivision (reservation/trust land): sum(tract pop x TractTribalArea area share) / county pop", "% of county population", "sram", S))
chr_years = {"chr_food_environment_index": ("Index 0 (worst)-10 (best) combining limited access to healthy foods and food insecurity", "index 0-10", "USDA FARA 2019 + MMG 2022"),
             "chr_pct_limited_access_healthy_food": ("Low-income population not living close to a grocery store", "%", "USDA FARA 2019"),
             "chr_pct_severe_housing_cost_burden": ("Households spending 50%+ of income on housing", "%", "ACS 2019-2023"),
             "chr_pct_children_single_parent": ("Children living in single-parent households", "%", "ACS 2019-2023"),
             "chr_pct_broadband": ("Households with broadband internet", "%", "ACS 2019-2023"),
             "chr_pct_uninsured_under65": ("Population under 65 without health insurance", "%", "SAHIE 2022"),
             "chr_pct_homeownership": ("Occupied housing units that are owner-occupied", "%", "ACS 2019-2023"),
             "chr_income_inequality_ratio": ("Household income 80th percentile / 20th percentile", "ratio", "ACS 2019-2023"),
             "chr_pct_children_frl_eligible": ("Public school children eligible for free or reduced-price lunch", "%", "NCES CCD 2022-23 school year (approx.)"),
             "chr_pct_child_care_cost_burden": ("Child care cost for 2 children as % of median household income", "%", "Living Institute/DOL 2024 + SAIPE 2023"),
             "chr_social_associations_per_10k": ("Membership associations per 10,000 population", "per 10k", "CBP 2022"),
             "chr_pct_census_participation": ("Households that self-responded to 2020 Census", "%", "Census 2020"),
             "chr_pct_long_commute_alone": ("Workers who drive alone with commute > 30 min", "%", "ACS 2019-2023"),
             "chr_pct_disability_functional": ("Adults with a functional disability", "%", "BRFSS/PLACES 2022"),
             "chr_pct_not_proficient_english": ("Population 5+ that speaks English less than 'well'", "%", "ACS 2019-2023"),
             "chr_pct_loneliness": ("Adults reporting feelings of loneliness", "%", "BRFSS/PLACES 2022 (modelled)")}
for c, (t, u, yrs) in chr_years.items():
    d.append(row(c, t, u, "chr", S, note=f"underlying data: {yrs}; check CHR 2025 technical notes"))
dd = pd.DataFrame(d)
assert set(dd.query("file == @Y").column) == set(yearly.columns) and set(dd.query("file == @S").column) == set(static.columns)
dd.to_csv(OUT / "county_factors_data_dictionary.csv", index=False)
print(len(dd), "dictionary rows")

68 dictionary rows


## 13 · Quick sanity: Spearman correlation of each new factor with MMG `fi_rate` 2024

Just a check that factors behave as expected (e.g. poverty ↑ with food insecurity). **No interpretation here**; analysis is in Q1 / Q6 notebooks.

In [15]:
from scipy.stats import spearmanr
fi = mmg.query("year == 2024")[["fips", "fi_rate"]]
y24 = yearly.query("year == 2024").drop(columns=["county", "year"]).filter(regex="^(?!.*_moe$)")
y23 = yearly.query("year == 2023")[["fips", "grocery_stores_per_10k", "food_bev_stores_per_10k"]].add_suffix("_2023").rename(columns={"fips_2023": "fips"})
m = fi.merge(y24, on="fips").merge(y23, on="fips").merge(static.drop(columns="county"), on="fips")
res = []
for c in m.columns.drop(["fips", "fi_rate"]):
    ok = m[[c, "fi_rate"]].dropna()
    if ok[c].dtype.kind in "if" and ok[c].nunique() > 2:
        r, p = spearmanr(ok[c], ok.fi_rate)
        res.append((c, len(ok), round(r, 2), round(p, 4)))
corr = pd.DataFrame(res, columns=["factor", "n", "spearman_rho", "p"]).sort_values("spearman_rho")
pd.set_option("display.max_rows", 100)
corr

,factor,n,spearman_rho,p
4,median_hh_income,87,-0.75,0.0000
41,chr_food_environment_index,87,-0.65,0.0000
52,chr_pct_census_participation,87,-0.55,0.0000
45,chr_pct_broadband,87,-0.44,0.0000
47,chr_pct_homeownership,87,-0.41,0.0001
53,chr_pct_long_commute_alone,87,-0.38,0.0002
0,pop_pep,87,-0.24,0.0249
37,sram_pct_lowinc_low_access_1_10,87,-0.20,0.0568
31,pct_asian,87,-0.16,0.1466
30,pct_black,87,-0.15,0.1801
